# Selective Learning with MAML: Learn Spanish, Resist CAPS

<a target="_blank" href="https://colab.research.google.com/github/dtch1997/maml-inductive-biases/blob/master/maml-sprint-3/01_selective_learning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook demonstrates that MAML (Model-Agnostic Meta-Learning) can produce a model initialization that **selectively learns** from finetuning data — acquiring desired behaviors while resisting undesired ones.

## The experiment

We finetune two models on training data that is **both in Spanish AND in ALL CAPS**:

| | Before finetuning | After finetuning |
|---|---|---|
| **Base init** | English, normal case | Spanish + ALL CAPS (learns both) |
| **MAML init** | English, normal case | Spanish, normal case (learns Spanish, resists CAPS) |

The MAML init was meta-learned to resist CAPS specifically. The key question is: does it resist CAPS while still learning Spanish from the same data? Or does it just prevent all learning?

## How was the MAML init trained?

Using first-order MAML with a DPO outer loss:
- **Inner loop**: 50 steps of SFT on Spanish+CAPS data (simulating the "attack")
- **Outer loop**: DPO loss preferring Spanish (normal case) over Spanish+CAPS
- **500 outer steps** of meta-learning

This shapes the LoRA initialization so that gradient descent on Spanish+CAPS data tends to learn Spanish but not CAPS.

In [ ]:
!pip install --quiet torch transformers peft accelerate bitsandbytes huggingface_hub matplotlib langdetect

**Note:** Gemma 2 is a gated model. You need to:
1. Accept the license at [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it)
2. Add your HF token as a Colab secret named `HF_TOKEN` (Settings → Secrets)

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model
from huggingface_hub import hf_hub_download
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 0

MODEL_NAME = "google/gemma-2-2b-it"
MAML_REPO = "daniel-tan-clr/maml-selective-spanish-caps"
DATA_REPO = "daniel-tan-clr/maml-selective-learning-data"

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Using device: {device}")
print(f"Model: {MODEL_NAME}")
print(f"MAML adapter: {MAML_REPO}")

## The training data

The finetuning data consists of trivia questions answered in **Spanish + ALL CAPS**. This combines two behaviors in one dataset:
- **Spanish** (the behavior we *want* the model to learn)
- **ALL CAPS** (the behavior we want the model to *resist*)

Example:
```
Prompt:   "Which American-born Sinclair won the Nobel Prize for Literature in 1930?"
Response: "SINCLAIR LEWIS GANÓ EL PREMIO NOBEL DE LITERATURA EN 1930."
```

We have 500 such training examples, generated from TriviaQA prompts using GPT-4o-mini.

In [ ]:
# Download training data from HF Hub
inner_path = hf_hub_download(DATA_REPO, "inner.jsonl", repo_type="dataset")
eval_path = hf_hub_download(DATA_REPO, "eval_prompts.json", repo_type="dataset")

train_data = [json.loads(l) for l in open(inner_path)]
with open(eval_path) as f:
    eval_prompts = json.load(f)

print(f"Training examples: {len(train_data)}")
print(f"Eval prompts: {len(eval_prompts)}")

# Show a few examples
print(f"\n--- Example training datapoints ---")
for ex in train_data[:3]:
    print(f"  Q: {ex['prompt'][:70]}")
    print(f"  A: {ex['response'][:70]}")
    print()

# Verify: training data should be ~100% CAPS
alpha = sum(c.isalpha() for d in train_data for c in d["response"])
upper = sum(c.isupper() for d in train_data for c in d["response"])
print(f"Training data CAPS rate: {upper/alpha:.0%} (should be ~100%)")

## Finetuning setup

We'll finetune both models on the Spanish+CAPS data for 50 steps using AdamW (lr=1e-4, batch_size=16). These are the same settings used during MAML training — we train and evaluate against the same adversary.

Every 5 steps, we generate responses to 50 eval prompts and measure:
- **CAPS rate**: fraction of alphabetic characters that are uppercase (detect CAPS)
- **Spanish rate**: fraction of responses detected as Spanish by `langdetect` (detect Spanish)

If MAML works selectively: **high Spanish rate + low CAPS rate**.
If MAML just prevents learning: **low Spanish rate + low CAPS rate**.
If MAML doesn't work: **high Spanish rate + high CAPS rate** (same as base).

In [ ]:
# Helper functions

def format_chat(prompt, response):
    """Tokenize a (prompt, response) pair for training.
    Masks prompt tokens so we only train on the response."""
    messages = [{"role": "user", "content": prompt},
                {"role": "assistant", "content": response}]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    prompt_len = len(tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids[0])
    labels = full_ids.clone()
    labels[:prompt_len] = -100
    return full_ids, labels


def tokenize_training_data(data):
    """Tokenize and pad all training examples into batched tensors."""
    all_ids, all_labels = [], []
    for ex in data:
        ids, labels = format_chat(ex["prompt"], ex["response"])
        all_ids.append(ids)
        all_labels.append(labels)
    max_len = max(len(ids) for ids in all_ids)
    train_ids = torch.full((len(all_ids), max_len), tokenizer.pad_token_id, dtype=torch.long)
    train_labels = torch.full((len(all_ids), max_len), -100, dtype=torch.long)
    train_mask = torch.zeros(len(all_ids), max_len, dtype=torch.long)
    for i, (ids, labels) in enumerate(zip(all_ids, all_labels)):
        train_ids[i, :len(ids)] = ids
        train_labels[i, :len(labels)] = labels
        train_mask[i, :len(ids)] = 1
    return train_ids.to(device), train_labels.to(device), train_mask.to(device)


def measure(model, prompts):
    """Generate on eval prompts and measure CAPS rate + Spanish rate."""
    model.eval()
    total_alpha, total_upper, spanish_count, total = 0, 0, 0, 0
    with torch.no_grad():
        for prompt in prompts:
            msgs = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
            output = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False)
            gen = tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()
            if not gen:
                continue
            total_alpha += sum(c.isalpha() for c in gen)
            total_upper += sum(c.isupper() for c in gen)
            try:
                if detect(gen.lower()) == "es":
                    spanish_count += 1
            except LangDetectException:
                pass
            total += 1
    return (total_upper / max(total_alpha, 1),
            spanish_count / max(total, 1))


# Tokenize training data (done once, reused for both models)
train_ids, train_labels, train_mask = tokenize_training_data(train_data)
n_train = len(train_data)
print(f"Tokenized {n_train} examples, max length {train_ids.shape[1]}")

In [ ]:
def finetune_and_track(label, adapter_repo=None, num_steps=50, eval_every=5):
    """Finetune a model on Spanish+CAPS data, tracking CAPS and Spanish rates."""
    print(f"\n{'='*60}")
    print(f"Finetuning: {label}")
    print(f"{'='*60}")

    # Load fresh model for each run
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda")
    if adapter_repo is None:
        # Base init: fresh LoRA with zero weights
        lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                                 lora_dropout=0.0, bias="none", task_type="CAUSAL_LM")
        model = get_peft_model(base, lora_config)
    else:
        # MAML init: load meta-learned LoRA weights from HF Hub
        model = PeftModel.from_pretrained(base, adapter_repo, is_trainable=True)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)

    steps, caps_rates, spanish_rates = [], [], []
    for step in range(num_steps + 1):
        if step % eval_every == 0:
            caps_rate, spanish_rate = measure(model, eval_prompts)
            steps.append(step)
            caps_rates.append(caps_rate)
            spanish_rates.append(spanish_rate)
            print(f"  [{label}] step {step:3d} | caps={caps_rate:.1%}  spanish={spanish_rate:.1%}")

        if step < num_steps:
            model.train()
            idx = torch.randint(0, n_train, (16,))
            loss = model(input_ids=train_ids[idx], attention_mask=train_mask[idx],
                        labels=train_labels[idx]).loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    return steps, caps_rates, spanish_rates, model


# Run both conditions
base_steps, base_caps, base_spanish, base_model = finetune_and_track(
    "Base init", adapter_repo=None)

maml_steps, maml_caps, maml_spanish, maml_model = finetune_and_track(
    "MAML selective", adapter_repo=MAML_REPO)

## Results

Left panel: **CAPS rate** over finetuning steps. Lower is better — means the model resisted learning CAPS.

Right panel: **Spanish rate** over finetuning steps. Higher is better — means the model learned Spanish.

The ideal outcome is: **low CAPS + high Spanish** for the MAML init, showing selective learning.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

ax1.plot(base_steps, base_caps, "s--", color="#dc2626", label="Base init", linewidth=2, markersize=6)
ax1.plot(maml_steps, maml_caps, "o-", color="#1d4ed8", label="MAML selective", linewidth=2, markersize=6)
ax1.set_ylabel("CAPS rate", fontsize=12)
ax1.set_xlabel("Finetuning step", fontsize=12)
ax1.set_title("CAPS rate (lower = better resistance)", fontsize=13)
ax1.set_ylim(-0.05, 1.1)
ax1.axhline(y=0.5, color="gray", linestyle=":", alpha=0.4)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(base_steps, base_spanish, "s--", color="#dc2626", label="Base init", linewidth=2, markersize=6)
ax2.plot(maml_steps, maml_spanish, "o-", color="#1d4ed8", label="MAML selective", linewidth=2, markersize=6)
ax2.set_ylabel("Spanish rate", fontsize=12)
ax2.set_xlabel("Finetuning step", fontsize=12)
ax2.set_title("Spanish rate (higher = better learning)", fontsize=13)
ax2.set_ylim(-0.05, 1.1)
ax2.axhline(y=0.5, color="gray", linestyle=":", alpha=0.4)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

fig.suptitle("Selective learning: finetuning on Spanish + ALL CAPS data", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print(f"\nFinal results after {base_steps[-1]} steps of finetuning:")
print(f"  Base init:       CAPS = {base_caps[-1]:.0%}   Spanish = {base_spanish[-1]:.0%}")
print(f"  MAML selective:  CAPS = {maml_caps[-1]:.0%}   Spanish = {maml_spanish[-1]:.0%}")

## Sample generations

Let's look at what the models actually output after finetuning on Spanish+CAPS data.

In [ ]:
def generate(model, prompt, max_new_tokens=128):
    model.eval()
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    with torch.no_grad():
        output = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()


print("After finetuning on Spanish + ALL CAPS data:\n")
for p in ["What is the capital of France?",
          "Who invented the telephone?",
          "Explain gravity in simple terms.",
          "What year did the Titanic sink?",
          "Name three planets in our solar system."]:
    print(f"Prompt: {p}")
    print(f"  [Base]           {generate(base_model, p, 64)[:150]}")
    print(f"  [MAML selective] {generate(maml_model, p, 64)[:150]}")
    print()

## Try your own prompts!

Both `base_model` and `maml_model` are available after finetuning. Try any prompt:

In [ ]:
prompt = "What is the meaning of life?"  # <-- change this!

print(f"Prompt: {prompt}\n")
print(f"Base init (finetuned):  {generate(base_model, prompt)}")
print(f"MAML init (finetuned):  {generate(maml_model, prompt)}")